# Exercise 5.4 — Two pairs of shaders

We now have **two shader programs**:

* **Program A** (as before): just `MVP` and the vertex colours.
* **Program B** (new): also receives a uniform `time`.
  * In the **vertex shader** the position is scaled: `s = 1 + 0.4·sin(2t)`, so the cube grows and shrinks.
  * In the **fragment shader** the colour is mixed with its opposite colour: `k = 0.5 + 0.5·sin(3t)`,
    `colour = mix(c, 1 − c, k)`, so the colours change smoothly.

In `render()` we choose, for each cube, which program to use with `gl.useProgram(...)`. Cubes marked
`pulse: true` use program B, the others use program A. The vertex buffers are still the same for all cubes.

Note: after `gl.useProgram` we must connect the attributes and look up the uniforms **of that program**, because
each program has its own locations.

## Library
Only **glm-js** is needed (for matrices). The WebGL code is written directly, so this notebook still works
even when the course JS server is down.

In [ ]:
%%html
<script src='https://git.io/glm-js.min.js'></script>

In [ ]:
%%html
<script id="ex54-vs" type="x-shader/x-vertex">
precision mediump float;
attribute vec3 in_Position;
attribute vec3 in_Color;
uniform mat4 MVP;
varying vec3 ex_Color;
void main(void) {
    gl_Position = MVP * vec4(in_Position, 1.0);
    ex_Color = in_Color;
}
</script>

In [ ]:
%%html
<script id="ex54-fs" type="x-shader/x-fragment">
precision mediump float;
varying vec3 ex_Color;
void main(void) {
    gl_FragColor = vec4(ex_Color, 1.0);
}
</script>

In [ ]:
%%html
<script id="ex54-pulse-vs" type="x-shader/x-vertex">
precision mediump float;
attribute vec3 in_Position;
attribute vec3 in_Color;
uniform mat4 MVP;
uniform float time;          // seconds, sent from JavaScript
varying vec3 ex_Color;
void main(void) {
    float s = 1.0 + 0.4 * sin(2.0 * time);            // scale between 0.6 and 1.4
    gl_Position = MVP * vec4(s * in_Position, 1.0);
    ex_Color = in_Color;
}
</script>

In [ ]:
%%html
<script id="ex54-pulse-fs" type="x-shader/x-fragment">
precision mediump float;
uniform float time;
varying vec3 ex_Color;
void main(void) {
    float k = 0.5 + 0.5 * sin(3.0 * time);            // between 0 and 1
    vec3 other = vec3(1.0) - ex_Color;                // the "opposite" colour
    gl_FragColor = vec4(mix(ex_Color, other, k), 1.0);
}
</script>

In [ ]:
%%html
<script>
// ---------- small WebGL helpers ----------
// compile the two shaders (read from the <script> tags) and link them into a program
window.makeProgram = function (gl, vsId, fsId) {
  const compile = (type, id) => {
    const s = gl.createShader(type);
    gl.shaderSource(s, document.getElementById(id).text);
    gl.compileShader(s);
    if (!gl.getShaderParameter(s, gl.COMPILE_STATUS)) console.error(id, gl.getShaderInfoLog(s));
    return s;
  };
  const p = gl.createProgram();
  gl.attachShader(p, compile(gl.VERTEX_SHADER, vsId));
  gl.attachShader(p, compile(gl.FRAGMENT_SHADER, fsId));
  gl.linkProgram(p);
  return p;
};
// copy a JS array of numbers into a GPU buffer (GL wants 32-bit floats)
window.makeBuffer = function (gl, data) {
  const b = gl.createBuffer();
  gl.bindBuffer(gl.ARRAY_BUFFER, b);
  gl.bufferData(gl.ARRAY_BUFFER, new Float32Array(data), gl.STATIC_DRAW);
  return b;
};
// connect a buffer (3 floats per vertex) to an attribute of the shader
window.useAttrib = function (gl, prog, name, buffer) {
  const loc = gl.getAttribLocation(prog, name);
  if (loc < 0) return;
  gl.bindBuffer(gl.ARRAY_BUFFER, buffer);
  gl.vertexAttribPointer(loc, 3, gl.FLOAT, false, 0, 0);
  gl.enableVertexAttribArray(loc);
};
// call render(time) on every animation frame
window.animate = function (render) {
  const step = (time) => { render(time); requestAnimationFrame(step); };
  requestAnimationFrame(step);
};
// ---------- cube: 6 faces x 2 triangles x 3 vertices = 36 vertices ----------
// only coordinates -1 and 1 are used; the 8 corners are repeated for every face
window.cubeData = function () {
  const p = [[-1,-1,-1], [ 1,-1,-1], [ 1, 1,-1], [-1, 1,-1],
             [-1,-1, 1], [ 1,-1, 1], [ 1, 1, 1], [-1, 1, 1]];
  // one different colour per corner (red, green, blue, yellow, ...)
  const c = [[1,0,0], [0,1,0], [0,0,1], [1,1,0], [1,0,1], [0,1,1], [1,0.5,0], [0.5,0.5,0.5]];
  const faces = [[4,5,6,7], [1,0,3,2], [0,4,7,3], [5,1,2,6], [7,6,2,3], [0,1,5,4]];
  const pos = [], col = [];
  for (const [a, b, cc, d] of faces)
    for (const i of [a, b, cc, a, cc, d]) {      // square face = 2 triangles
      pos.push(...p[i]);
      col.push(...c[i]);
    }
  return { pos, col, n: pos.length / 3 };
};
</script>

In [ ]:
%%html
<canvas id="ex54Canvas" width="500" height="500" style="border:2px solid #000000;"></canvas>
<div id="ex54CanvasInfo" style="font-family:monospace"></div>
<script>
(function () {
  const cv = document.getElementById("ex54Canvas");
  const gl = cv.getContext("webgl");
  const progA = makeProgram(gl, "ex54-vs", "ex54-fs");               // normal
  const progB = makeProgram(gl, "ex54-pulse-vs", "ex54-pulse-fs");   // scale + colour vary with time
  const cube = cubeData();
  const posBuf = makeBuffer(gl, cube.pos), colBuf = makeBuffer(gl, cube.col);
  gl.enable(gl.DEPTH_TEST);
  gl.clearColor(1, 1, 1, 1);

  // each cube: position + rotation axis + speed. All use the SAME two buffers.
  const cubes = [
    { pos: [-3,  0,  0], axis: [1, 0, 0], speed: 1.0 },
    { pos: [ 0,  0,  0], axis: [0, 1, 0], speed: 0.7 },
    { pos: [ 3,  0,  0], axis: [0, 0, 1], speed: 1.3 },
    { pos: [-1.5, 2.5, -2], axis: [1, 1, 0], speed: 0.9 },
    { pos: [ 1.5, -2.5, -2], axis: [0, 1, 1], speed: 1.6 },
  ];

  cubes[1].pulse = true;  cubes[3].pulse = true;   // these two use program B

  const V = glm.lookAt(glm.vec3(0, 3, 12), glm.vec3(0, 0, 0), glm.vec3(0, 1, 0));
  const P = glm.perspective(glm.radians(45), cv.width / cv.height, 0.1, 100);

  animate((time) => {
    gl.clear(gl.COLOR_BUFFER_BIT | gl.DEPTH_BUFFER_BIT);
    for (const c of cubes) {
      const prog = c.pulse ? progB : progA;
      gl.useProgram(prog);
      useAttrib(gl, prog, "in_Position", posBuf);
      useAttrib(gl, prog, "in_Color", colBuf);
      if (c.pulse) gl.uniform1f(gl.getUniformLocation(prog, "time"), time / 1000);

      const M = glm.translate(glm.mat4(1), glm.vec3(...c.pos))['*'](
                glm.rotate(glm.mat4(1), c.speed * time / 1000, glm.normalize(glm.vec3(...c.axis))));
      gl.uniformMatrix4fv(gl.getUniformLocation(prog, "MVP"), false, P['*'](V['*'](M)).elements);
      gl.drawArrays(gl.TRIANGLES, 0, cube.n);
    }
    document.getElementById("ex54CanvasInfo").textContent = "cubes 2 and 4 use the second shader pair";
  });
})();
</script>